# Phase 4: Training Pipeline (unet1d)

In [21]:
# # =========================================================
# # CELL -1: THE "NUKE" (CLEAN SLATE PROTOCOL)
# # =========================================================
# import os
# import time

# print("🧹 Initiating System Purge (Protecting your files & keys)...")

# # 1. TERMINATE ZOMBIES
# # Kill any background rclone or gcsfuse processes holding the OS hostage
# !pkill -9 rclone > /dev/null 2>&1
# !pkill -9 gcsfuse > /dev/null 2>&1

# # 2. FORCE UNMOUNT (THE LAZY WAY)
# # The '-uz' flag tells Linux: "Unmount this immediately, even if a process is using it."
# # This is completely safe for the cloud files; it just breaks the local OS link.
# !fusermount -uz /content/drive/MyDrive/seismic-first-break-picking > /dev/null 2>&1
# !fusermount -uz /content/drive/MyDrive > /dev/null 2>&1
# !fusermount -uz /content/drive > /dev/null 2>&1

# # 3. NUKE CORRUPT CONFIGS
# # Delete any half-written or broken rclone configs from previous failed runs.
# # (Your credentials.json remains completely untouched).
# !rm -f /root/.config/rclone/rclone.conf

# print("⏳ Letting the OS clear the memory buffers...")
# time.sleep(3) # Give the Linux kernel a moment to breathe

# print("✅ System is completely clean and reset.")
# print("➡️ You may now run Cell 0.")
# print("-" * 50)

In [22]:
# =========================================================
# CELL 0: STABLE BRIDGE (MODERN RCLONE)
# =========================================================
import os
import sys
import time
from types import ModuleType

print("🚀 Initializing Stable Service Account Bridge...")

# --- USER CONFIGURATION ---
FOLDER_ID = "1SQmj0nT__IJGPYLs--uo5pa_vPo1w8bS"
KEY_PATH = "/credentials.json"
target_dir = '/content/drive/MyDrive/seismic-first-break-picking'
# --------------------------

# 1. PURGE DEAD MOUNTS
!pkill -9 rclone > /dev/null 2>&1
!fusermount -uz {target_dir} > /dev/null 2>&1

# 2. INSTALL MODERN RCLONE (Bypassing old apt-get versions)
print("📦 Installing latest Rclone engine...")
!curl -s https://rclone.org/install.sh | sudo bash > /dev/null 2>&1

# 3. WRITE CONFIG
if not os.path.exists(KEY_PATH):
    print(f"❌ ERROR: Cannot find {KEY_PATH}.")
else:
    print("🔑 Authenticating...")
    os.makedirs('/root/.config/rclone', exist_ok=True)
    rclone_conf = f"""
[gdrive]
type = drive
scope = drive
service_account_file = {KEY_PATH}
root_folder_id = {FOLDER_ID}
"""
    with open('/root/.config/rclone/rclone.conf', 'w') as f:
        f.write(rclone_conf)

    # 4. MOUNT WITH SMART-WAIT
    os.makedirs(target_dir, exist_ok=True)
    print("🛰️ Mounting Project...")

    # Using os.system to safely detach the background process
    os.system(f"rclone mount gdrive: {target_dir} --daemon --vfs-cache-mode writes --allow-non-empty")

    print("⏳ Waiting for OS to lock in the mount...")
    # Smart polling: We check every second until the files physically appear
    for _ in range(15):
        try:
            files = os.listdir(target_dir)
            if len(files) > 0:
                print(f"✅ SUCCESS: Project live! Found: {', '.join(files[:4])}...")
                break
        except OSError:
            pass # Ignore temporary Errno 5 while it wakes up
        time.sleep(1)
    else:
        print("❌ ERROR: Mount timed out. Run the debug cell again.")

# 5. DISARM CELL 1
colab_mock = ModuleType('google.colab')
drive_mock = ModuleType('drive')
drive_mock.mount = lambda *args, **kwargs: print("✨ drive.mount() bypassed: Using Secret Key bridge.")
colab_mock.drive = drive_mock
sys.modules['google.colab'] = colab_mock
sys.modules['google.colab.drive'] = drive_mock

print("\n🛡️ Cell 1 is disarmed. Hit 'Run All'.")
print("-" * 50)

🚀 Initializing Stable Service Account Bridge...
📦 Installing latest Rclone engine...
🔑 Authenticating...
🛰️ Mounting Project...
⏳ Waiting for OS to lock in the mount...
✅ SUCCESS: Project live! Found: .git, .gitignore, .tmp.drivedownload, .tmp.driveupload...

🛡️ Cell 1 is disarmed. Hit 'Run All'.
--------------------------------------------------


In [23]:
# =========================================================
# CELL 0.5: EMERGENCY RESET & STABLE PATCHER
# =========================================================
import os
import sys
import importlib
import pandas as pd

print("🧹 Cleaning up dirty module memory...")

# 1. THE "NUCLEAR" RESET
# We force Python to forget the patched version ever existed
to_delete = [m for m in sys.modules if m.startswith('src.utils.config_loader')]
for m in to_delete:
    del sys.modules[m]

# 2. RE-IMPORT FRESH FROM DRIVE
# This ensures we are grabbing the CLEAN code from the file, not the broken one in RAM
import src.utils.config_loader as cl
importlib.reload(cl)
print("✅ Clean config_loader reloaded from Drive.")

# 3. THE SMART DATA CACHE
local_data = '/content/local_data'
if os.path.exists(local_data) and len(os.listdir(local_data)) > 0:
    print(f"✅ Fast cache already exists. Skipping copy.")
else:
    print(f"📦 Copying dataset to ultra-fast NVMe...")
    os.makedirs(local_data, exist_ok=True)
    os.system(f"cp -a /content/drive/MyDrive/seismic-first-break-picking/data/. {local_data}/")
    print("✅ Data successfully cached!")

# 4. THE RECURSION-PROOF PATCH
# We capture the TRULY clean function now
clean_loader = cl.load_model_config

def patched_loader(path, *args, **kwargs):
    # This calls the CLEAN version we just imported
    config = clean_loader(path, *args, **kwargs)

    print("\n✨ Intercepting YAML load (Local NVMe Mode)...")

    # CSV REDIRECTION
    if hasattr(config, 'data') and hasattr(config.data, 'index_csv'):
        old_csv_path = config.data.index_csv if os.path.isabs(config.data.index_csv) else \
                       os.path.join('/content/drive/MyDrive/seismic-first-break-picking', config.data.index_csv)

        local_csv_path = os.path.join(local_data, 'nvme_patched_index.csv')
        if os.path.exists(old_csv_path):
            df = pd.read_csv(old_csv_path)
            df = df.replace(r'/content/drive/MyDrive/.*?/data/', '/content/local_data/', regex=True)
            df.to_csv(local_csv_path, index=False)
            config.data.index_csv = local_csv_path
            print(f"   -> Redirected index_csv to NVMe")

    # PATH REDIRECTION
    def patch_namespace(ns):
        for key, value in vars(ns).items():
            if isinstance(value, str) and ('data' in value.lower()):
                if not any(x in key.lower() for x in ['model', 'checkpoint', 'output', 'csv']):
                    setattr(ns, key, local_data)
                    print(f"   -> Redirected config.{key} to {local_data}")
            elif type(value).__name__ == 'SimpleNamespace':
                patch_namespace(value)

    patch_namespace(config)
    return config

# Apply the patch to the module
cl.load_model_config = patched_loader

print("✅ Stable patch applied. NO MORE RECURSION.")
print("-" * 50)

🧹 Cleaning up dirty module memory...
✅ Clean config_loader reloaded from Drive.
✅ Fast cache already exists. Skipping copy.
✅ Stable patch applied. NO MORE RECURSION.
--------------------------------------------------


In [24]:
# Cell 1: Environment Setup & Hardware Detection
import os
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def find_project_root(indicator_file="configs/datasets.yaml"):
    # Dynamically find the project root relative to the notebook location.
    curr = os.getcwd()
    # Check current and up to 4 parent levels (Notebooks are usually in notebooks/)
    for _ in range(5):
        if os.path.exists(os.path.join(curr, indicator_file)):
            return curr
        curr = os.path.dirname(curr)
    # Fallback for local dev or standard Colab location
    return '/content/drive/MyDrive/seismic-first-break-picking'

PROJECT_ROOT = find_project_root()

if IN_COLAB:
    # Ensure all required packages are installed in the fresh Colab runtime
    !pip install -q mlflow optuna lightgbm segmentation-models-pytorch pyyaml

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

import torch
import numpy as np
import random
import matplotlib.pyplot as plt

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device_type = 'cuda' if torch.cuda.is_available() else 'cpu'
device = torch.device(device_type)
device_name = torch.cuda.get_device_name(0) if device_type == 'cuda' else 'cpu'
vram_bytes = torch.cuda.get_device_properties(0).total_memory if device_type == 'cuda' else 0

print(f"Device: {device_name}")
print(f"VRAM: {vram_bytes / 1e9:.2f} GB" if vram_bytes > 0 else "")


✨ drive.mount() bypassed: Using Secret Key bridge.
Device: NVIDIA L4
VRAM: 23.66 GB


In [25]:
# Cell 2: Config & MLflow Setup
import mlflow
import types
from src.utils.config_loader import load_model_config
from src.training.mlflow_logger import MLFlowLogger

config_path = os.path.join(PROJECT_ROOT, 'configs', 'model_unet1d.yaml')
config = load_model_config(config_path, as_namespace=True)

# Defensive fallback: guarantee config.output exists with all required keys
if not hasattr(config, 'output'):
    config.output = types.SimpleNamespace()
if not hasattr(config.output, 'tracking_uri'):
    config.output.tracking_uri = f'file:///{PROJECT_ROOT}/mlruns'
if not hasattr(config.output, 'checkpoint_dir'):
    config.output.checkpoint_dir = f'models/unet1d'

# CRITICAL: Resolve checkpoint_dir to an absolute path inside Google Drive.
# If it is a relative path (e.g. 'models/cnn1d'), Colab resolves it against
# CWD=/content (ephemeral disk) � models would be LOST when the session ends.
# Prepend PROJECT_ROOT so all checkpoints land on Drive unconditionally.
if not os.path.isabs(config.output.checkpoint_dir):
    config.output.checkpoint_dir = os.path.join(PROJECT_ROOT, config.output.checkpoint_dir)
os.makedirs(config.output.checkpoint_dir, exist_ok=True)
print(f"Checkpoint dir (absolute): {config.output.checkpoint_dir}")

# Terminate any active MLflow run from a previous cell execution (rerun safety)
mlflow.end_run()

# --- Resume-safe MLflow init ---
# _save_checkpoint writes mlflow_run_id.txt on every epoch save.
# If the file exists, we rejoin that run so all epochs appear on one timeline.
# If the run was FINISHED (or the file is stale) resume_run() transparently
# falls back to a fresh run, so re-running a completed notebook is fully safe.
_run_id_file = os.path.join(config.output.checkpoint_dir, 'mlflow_run_id.txt')
_existing_run_id = None
if os.path.exists(_run_id_file):
    with open(_run_id_file) as _f:
        _existing_run_id = _f.read().strip() or None

logger = MLFlowLogger(config.output.tracking_uri, config.experiment.name)
if _existing_run_id:
    logger.resume_run(_existing_run_id)
    print(f"Resumed MLflow run: {_existing_run_id}")
else:
    logger.start_run()

logger.log_params(config)
print(f"Loaded config for: {config.experiment.name}")
print(f"Checkpoint dir: {config.output.checkpoint_dir}")



✨ Intercepting YAML load (Local NVMe Mode)...
   -> Redirected index_csv to NVMe
   -> Redirected config.processed_dir to /content/local_data
Checkpoint dir (absolute): /content/drive/MyDrive/seismic-first-break-picking/models/unet1d
Resumed MLflow run: 35829fc149c84259bd01e344e6aaa584
Loaded config for: unet1d_softargmax
Checkpoint dir: /content/drive/MyDrive/seismic-first-break-picking/models/unet1d


In [26]:
# Cell 3: Checkpoint State Detection
import os
import torch

# Trainer saves as '{experiment_name}_latest.pt' � match that filename exactly.
checkpoint_path = os.path.join(config.output.checkpoint_dir, f'{config.experiment.name}_latest.pt')
is_progressive = hasattr(config, 'progressive_training') and getattr(config.training, 'training_mode', 'combined') == 'progressive'

if not os.path.exists(checkpoint_path):
    state = 'NO_CHECKPOINT_EXISTS'
    start_asset_index = 0
    resume_epoch = 0
    completed_assets = []
else:
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    training_state = checkpoint.get('training_state', {})
    resume_epoch = checkpoint.get('epoch', 0)

    if is_progressive:
        if training_state.get('is_fully_trained', False):
            state = 'TRAINING_COMPLETE'
            completed_assets = config.progressive_training.asset_order
            start_asset_index = len(completed_assets)
        else:
            completed_assets = training_state.get('completed_assets', [])
            start_asset_index = len(completed_assets)
            state = f"ASSET_{start_asset_index}_COMPLETE" if completed_assets else "NO_CHECKPOINT_EXISTS"
    else:
        # Combined Training (Default)
        completed_assets = []
        start_asset_index = 0
        state = 'RESUMING_COMBINED_TRAINING'

print(f"Detected state: {state}")
print(f"Resuming from epoch: {resume_epoch}")


Detected state: RESUMING_COMBINED_TRAINING
Resuming from epoch: 6


In [27]:
# Cell 4: Data & Sampler Setup
from torch.utils.data import DataLoader
from src.data.dataset import ShotGatherDataset, ProgressiveAssetSampler, variable_width_collate_fn, trace_collate_fn
from src.data.transforms import build_transforms
from src.utils.config_loader import load_yaml

csv_path = config.data.index_csv if os.path.isabs(config.data.index_csv) else os.path.join(PROJECT_ROOT, config.data.index_csv)

# Load augmentation config and build training transforms
preproc_path = os.path.join(PROJECT_ROOT, 'configs', 'preprocessing.yaml')
preproc_cfg = load_yaml(preproc_path)
train_transform = build_transforms(preproc_cfg.get('augmentation', {}), is_training=True)
print(f"Train augmentations: {train_transform}")

train_ds = ShotGatherDataset(csv_path, split='train', transform=train_transform)
val_ds = ShotGatherDataset(csv_path, split='val')  # No augmentation for validation

val_loader = DataLoader(
    val_ds, batch_size=config.training.batch_size, shuffle=False,
    collate_fn=trace_collate_fn, num_workers=2, pin_memory=True
)

train_loader = DataLoader(
    train_ds, batch_size=config.training.batch_size, shuffle=True,
    collate_fn=trace_collate_fn, num_workers=2, pin_memory=True
)
print(f"Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")


Train augmentations: Compose([
])
Train samples: 2907 | Val samples: 621


In [28]:
# Cell 5: Model Map
from src.models.unet_1d import SoftArgmax1DUNet
model = SoftArgmax1DUNet(base_channels=getattr(config.model, 'base_channels', 32)).to(device)
print(f"Parameters: {model.count_parameters()}")

Parameters: 2707809


In [29]:
# Cell 6: Phase 4.7 Optuna Sweeper (Optional: Run ONLY on final best architecture)
try:
    import optuna
    from optuna.pruners import MedianPruner
except ImportError:
    optuna = None
    print("Install optuna for sweeps.")

def run_optuna_sweep(n_trials=30):
    if optuna is None: return

    print("WARNING: Do not run Optuna sweeps on all models. Compute limits apply.")
    print("Execute this function manually only on the singular best architecture.")

    def objective(trial):
        lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
        bs = trial.suggest_categorical("batch_size", [4, 8, 16] if '2d' in model_key else [32, 64, 128])
        # Insert discrete model instantiations + train epochs.
        # Report intermediate validation MAE to the pruner at each epoch:
        # trial.report(val_mae, epoch)
        # if trial.should_prune():
        #     raise optuna.TrialPruned()
        return 0.0 # Return absolute Val MAE here.

    study = optuna.create_study(direction="minimize", pruner=MedianPruner(n_warmup_steps=10))
    study.optimize(objective, n_trials=n_trials)
    print("Best params:", study.best_params)


In [30]:
# Cell 7: Execute Progressive Loop
from src.models.loss import MaskedMAELoss
from src.training.trainer import Trainer

criterion = MaskedMAELoss()

if state == 'TRAINING_COMPLETE':
    print("Skipping to evaluation")
else:
    # We iterate over the sequence of assets dynamically
    # For Combined mode, asset_order just has 'all'
    is_progressive = hasattr(config, 'progressive_training') and getattr(config.training, 'training_mode', 'combined') == 'progressive'

    if not is_progressive:
        optimizer = torch.optim.AdamW(model.parameters(), lr=config.training.lr)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.training.epochs)
        trainer = Trainer(model, optimizer, criterion, config, device, scheduler, logger)
        # Resume from Drive checkpoint if one was detected in Cell 3
        if state == 'RESUMING_COMBINED_TRAINING' and os.path.isfile(checkpoint_path):
            trainer.load_checkpoint(checkpoint_path)
            print(f"Resumed from checkpoint at epoch {trainer.start_epoch - 1}")
        trainer.run(train_loader, val_loader)
    else:
        # Loop Progressive Assets
        for i in range(start_asset_index, len(config.progressive_training.asset_order)):
            current_asset = config.progressive_training.asset_order[i]
            print(f"--- Training Phase: {current_asset} ---")

            p_sampler = ProgressiveAssetSampler(
                train_ds,
                current_asset=current_asset,
                previous_assets=config.progressive_training.asset_order[:i],
                replay_fraction=config.progressive_training.replay_fraction
            )

            p_loader = DataLoader(
                train_ds, batch_size=config.training.batch_size, sampler=p_sampler,
                collate_fn=trace_collate_fn, num_workers=2, pin_memory=True
            )

            optimizer = torch.optim.AdamW(model.parameters(), lr=config.progressive_training.asset_lr[i])
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.progressive_training.asset_epochs[i])

            trainer = Trainer(model, optimizer, criterion, config, device, scheduler, logger)

            # On the first resumable asset, reload model weights from the checkpoint.
            # (The optimizer is intentionally NOT restored � each asset starts a fresh
            # optimiser phase with its own LR.  Only the learned weights carry over.)
            if i == start_asset_index and resume_epoch > 0 and os.path.isfile(checkpoint_path):
                _ckpt = torch.load(checkpoint_path, map_location=device)
                model.load_state_dict(_ckpt['model_state_dict'])
                print(f"  Loaded model weights from checkpoint (epoch {resume_epoch})")

            trainer.run(p_loader, val_loader,
                        start_epoch=resume_epoch,
                        total_epochs=config.progressive_training.asset_epochs[i])

            # Mark this asset complete inside the trainer so _save_checkpoint
            # persists the completed list to disk � prevents progressive amnesia
            # if the session dies before all assets finish.
            trainer.training_state = {
                'completed_assets': config.progressive_training.asset_order[:i + 1],
                'is_fully_trained': (i + 1) == len(config.progressive_training.asset_order),
            }
            trainer._save_checkpoint(
                config.progressive_training.asset_epochs[i], False,
                f"{config.experiment.name}_latest.pt"
            )

            # Reset resume so next asset starts at epoch 1
            resume_epoch = 0


Loading checkpoint from /content/drive/MyDrive/seismic-first-break-picking/models/unet1d/unet1d_softargmax_latest.pt
Resumed from epoch 6 with Best MAE: 152.1825
  Restored 6 history entries.
Resumed from checkpoint at epoch 6

--- Epoch 7/30 ---


Training:   1%|          | 18/2907 [00:06<19:11,  2.51it/s, loss=120]/content/drive/MyDrive/seismic-first-break-picking/src/models/loss.py:31: RuntimeWarning: MaskedMAELoss received a batch with NO valid labels. Returning 0.0 loss with gradient attached.
  warnings.warn("MaskedMAELoss received a batch with NO valid labels. Returning 0.0 loss with gradient attached.", RuntimeWarning)
Training: 100%|██████████| 2907/2907 [11:55<00:00,  4.06it/s, loss=71.7]


Train | Loss: 112.5155 | MAE: 140.38ms
Val   | Loss: 146.9945 | MAE: 154.51ms | In-5ms: 2.20%

--- Epoch 8/30 ---


Training: 100%|██████████| 2907/2907 [12:04<00:00,  4.01it/s, loss=160]


Train | Loss: 111.9114 | MAE: 140.17ms
Val   | Loss: 134.2072 | MAE: 152.95ms | In-5ms: 2.47%

--- Epoch 9/30 ---


...

Train | Loss: 105.9685 | MAE: 134.93ms


Val   | Loss: 143.1565 | MAE: 154.06ms | In-5ms: 2.26%

--- Epoch 30/30 ---


Training: 100%|██████████| 2907/2907 [12:05<00:00,  4.01it/s, loss=224]

Train | Loss: 105.8468 | MAE: 134.86ms


Val   | Loss: 152.3734 | MAE: 158.22ms | In-5ms: 2.00%


In [31]:
# Cell 8: Universal Evaluation (all model tiers)
# Force-purge ALL src.* modules from Colab's module cache so that any fixes
# pushed to Drive (loss.py, trainer.py, dataset.py, evaluator.py, etc.) are
# reloaded fresh. Without this, Colab reuses stale bytecode from the session's
# first import, silently ignoring edits on Drive.
import importlib, sys
_stale = [k for k in sys.modules if k.startswith('src.')]
for _mod_name in _stale:
    del sys.modules[_mod_name]
print(f"Cache cleared: {len(_stale)} src.* modules evicted.")

from src.training.evaluator import ModelEvaluator

_is_dl = 'unet1d' not in ('classical', 'tabular_lgbm')
_history = trainer.history if 'trainer' in dir() and hasattr(trainer, 'history') else {}

evaluator = ModelEvaluator(
    model=model,
    val_loader=val_loader,
    logger=logger,
    device=device,
    model_key='unet1d',
    is_dl=_is_dl,
    history=_history,
)
final_metrics = evaluator.run()

# Close MLflow run cleanly
import mlflow
mlflow.end_run()
print("MLflow run closed.")


Cache cleared: 14 src.* modules evicted.

MODEL EVALUATOR: unet1d

──────────────────────────────────────────────────
  VALIDATION METRICS
──────────────────────────────────────────────────
  MAE (mean):      158.23 ms
  MAE P50:         133.67 ms
  MAE P90:         314.80 ms
  MAE P95:         395.05 ms
  Within   5ms:    2.0%
  Within  10ms:    4.0%
  Within  20ms:    8.1%
  Parameters:      2,707,809
  Latency/trace:   0.098 ms
  Throughput:      10199 traces/s

  PER-ASSET BREAKDOWN
  brunswick    | MAE:  174.3 ms | <=5ms:   1.8%  (553838 traces)
  halfmile     | MAE:  117.8 ms | <=5ms:   2.6%  (145461 traces)
  lalor        | MAE:  132.1 ms | <=5ms:   2.5%  (184362 traces)
  sudbury      | MAE:  218.6 ms | <=5ms:   0.1%  (29335 traces)
──────────────────────────────────────────────────

All metrics and plots logged to MLflow.
MLflow run closed.


In [32]:
# Move the ephemeral logs to your permanent Drive folder
!cp -r /content/mlruns /content/drive/MyDrive/seismic-first-break-picking/
print("✅ Logs successfully moved to Google Drive!")

✅ Logs successfully moved to Google Drive!


In [33]:
!zip -r mlruns_backup.zip /content/mlruns

  adding: content/mlruns/ (stored 0%)
  adding: content/mlruns/.trash/ (stored 0%)
  adding: content/mlruns/0/ (stored 0%)
  adding: content/mlruns/0/meta.yaml (deflated 30%)
  adding: content/mlruns/520174565831125385/ (stored 0%)
  adding: content/mlruns/520174565831125385/af7830603eec4a11852a426a0e49d3f4/ (stored 0%)
  adding: content/mlruns/520174565831125385/af7830603eec4a11852a426a0e49d3f4/metrics/ (stored 0%)
  adding: content/mlruns/520174565831125385/af7830603eec4a11852a426a0e49d3f4/params/ (stored 0%)
  adding: content/mlruns/520174565831125385/af7830603eec4a11852a426a0e49d3f4/params/training.lr (stored 0%)
  adding: content/mlruns/520174565831125385/af7830603eec4a11852a426a0e49d3f4/params/experiment.seed (stored 0%)
  adding: content/mlruns/520174565831125385/af7830603eec4a11852a426a0e49d3f4/params/model.architecture (stored 0%)
  adding: content/mlruns/520174565831125385/af7830603eec4a11852a426a0e49d3f4/params/model.base_channels (stored 0%)
  adding: content/mlruns/5201745